# 7-eonmol · 端到端可复现演示

本 Notebook 用于证明提交的代码**在上述环境中可实际执行**，覆盖关键流程：

1. 环境与目录结构自检
2. 模型权重加载（含 PyTorch 2.6 兼容补丁）
3. 靶点登记表读取（口袋中心如何确定）
4. 口袋引导束搜索生成（小规模，便于快速复核）
5. 候选分子的类药性与新颖性核验
6. 一键入口（`predict.py` / `train.py`）的命令速查

运行前提：
- 已按 `requirements.txt` 安装依赖，当前工作目录为**仓库根**
- 有 NVIDIA GPU（无 GPU 时把 `DEVICE` 改成 `"cpu"`，速度显著变慢）
- 第 4 步为小规模演示（5 分子 / 束宽 20 / 步数上限 10），数分钟内可完成


In [ ]:
import os, sys, platform, subprocess

# 让 src/ 可导入（与所有入口脚本一致的做法）
ROOT = os.path.abspath(os.path.join(os.getcwd()))
sys.path.insert(0, os.path.join(ROOT, "src"))
from paths import (ROOT as PROJ, SRC, DATA, MODELS, CONFIGS, RESULTS, OUTPUTS,
                   TARGETS, KNOWN_DRUGS, OTHER, PRETRAINED, PYTHON)

print("仓库根 :", PROJ)
print("Python :", sys.version.split()[0], "|", platform.platform())
print("解释器 :", PYTHON)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch  :", torch.__version__, "| CUDA 可用:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("使用设备:", DEVICE)


In [ ]:
# 目录结构自检：模板要求的板块是否齐备
need = {
    "README.md": os.path.join(PROJ, "README.md"),
    "requirements.txt": os.path.join(PROJ, "requirements.txt"),
    "src/": SRC, "data/": DATA, "models/": MODELS, "notebooks/": os.path.join(PROJ, "notebooks"),
    "results/": RESULTS, "logs/": os.path.join(PROJ, "logs"),
    "train.py": os.path.join(PROJ, "train.py"),
    "design.py": os.path.join(PROJ, "design.py"),
    "predict.py": os.path.join(PROJ, "predict.py"),
    "预训练权重": PRETRAINED,
    "靶点登记表": os.path.join(CONFIGS, "targets.json"),
}
bad = [k for k, v in need.items() if not os.path.exists(v)]
for k, v in need.items():
    print(("  [OK]  " if os.path.exists(v) else "  [缺失] ") + k)
assert not bad, "缺少: %s" % bad
print("\n目录结构自检通过")


## 2. 模型权重加载

权重为 Pocket2Mol 官方预训练权重（MIT，(c) 2022 Xingang Peng）。
PyTorch 2.6 起 `torch.load` 默认 `weights_only=True`，本仓库在
`src/utils/misc.py` 中注册了安全全局对象（补丁），因此需先 `import utils.misc`。


In [ ]:
from utils import misc  # 触发 PyTorch 2.6 的 add_safe_globals 补丁

ck = torch.load(PRETRAINED, map_location="cpu")
n_par = sum(v.numel() for v in ck["model"].values()) if isinstance(ck, dict) and "model" in ck else None
print("权重文件:", os.path.relpath(PRETRAINED, PROJ), "(%.1f MB)" % (os.path.getsize(PRETRAINED) / 1e6))
print("顶层键  :", list(ck.keys()) if isinstance(ck, dict) else type(ck).__name__)
print("参数量  :", "%.2f M" % (n_par / 1e6) if n_par else "n/a")
print("迭代数  :", ck.get("iteration") if isinstance(ck, dict) else "n/a")


## 3. 靶点登记表（口袋中心怎么来的）

`configs/targets.json` 由 `src/scripts/build_target_registry.py` **从结构文件本身**生成：
受体名取自结构条目标题，共晶配体与口袋中心取自该配体的**实验重原子坐标质心**。
其中"沉积配体完整性"用于披露结构质量问题（如 A1AR 的共晶配体缺一个氟原子）。


In [ ]:
import json
reg = json.load(open(os.path.join(CONFIGS, "targets.json"), encoding="utf-8"))
print("%-8s %-6s %-6s %-8s %s" % ("靶点", "PDB", "配体", "中心(Å)", "受体"))
for name, t in reg.items():
    pdb = os.path.basename(t["pdb"])
    print("%-8s %-6s %-6s %-8s %s" % (
        name, pdb.split("_")[0], t["ligand_comp"],
        ",".join("%.1f" % v for v in t["center"]), t["receptor"][:46]))


## 4. 口袋引导束搜索生成（小规模演示）

引导打分（只改变采样分布，不改变网络结构或权重）：

```
P ∝ (exp(Σ logP_model) + 1) · w · exp( λ · [ QED + (1 − SA/10) − w_div · maxTanimoto ] )
```

- `λ=3` 为实测折中点；`λ=0` 即退化为原版无引导采样（可用于 A/B 对照）
- 生成成本由**束宽**主导：束宽越大越慢；步数是**安全上限**而非目标值

下面调用仓库的一键入口 `design.py`，用很小的规模跑通链路。


In [ ]:
TARGET, NUM, BEAM, STEPS, SEED = "A2A", 5, 20, 10, 2024
DEMO_OUT = os.path.join(OUTPUTS, "notebook_demo")

cmd = [PYTHON, os.path.join(PROJ, "design.py"),
       "--target", TARGET, "--num-samples", str(NUM), "--beam", str(BEAM),
       "--max-steps", str(STEPS), "--seed", str(SEED), "--lam", "3.0",
       "--device", DEVICE, "--outdir", DEMO_OUT]
print("运行:", " ".join(cmd[-12:]))
r = subprocess.run(cmd, cwd=PROJ)
print("\n返回码:", r.returncode)
assert r.returncode == 0, "生成失败，请查看上方输出"


In [ ]:
import glob
# 找到本次生成的会话目录（design.py 会自建 <outdir>/<target>_<时间戳>/）
sessions = sorted(d for d in glob.glob(os.path.join(DEMO_OUT, "*")) if os.path.isdir(d))
sess = sessions[-1]
# sample_for_pdb 会再建一层会话目录
inner = sorted(d for d in glob.glob(os.path.join(sess, "*")) if os.path.isdir(d))
sess = inner[-1] if inner else sess
smi_file = os.path.join(sess, "SMILES.txt")
sdfs = glob.glob(os.path.join(sess, "SDF", "*.sdf"))
smis = sorted({s.strip() for s in open(smi_file, encoding="utf-8") if s.strip()}) if os.path.exists(smi_file) else []
print("会话目录:", os.path.relpath(sess, PROJ))
print("SDF 文件: %d 个" % len(sdfs))
print("唯一 SMILES: %d 个" % len(smis))


## 5. 候选核验：类药性、合成可及性、新颖性

指标口径与 `predict.py`、`src/evaluation/scoring_func.py` 一致：
类药性综合分 = `QED + (1 − SA/10)`（即生成阶段的引导目标）；
新颖性 = 与靶点已知活性分子的**最大 Tanimoto 相似度**（越低越新）。


In [ ]:
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Descriptors, Crippen, rdMolDescriptors, AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import FilterCatalog
RDLogger.DisableLog("rdApp.*")

SA_CALC = None
try:
    from evaluation.sascorer import calculateScore as SA_CALC   # 仓库自带 SA 打分器
except Exception as e:
    print("SA 打分器不可用(%s)，将跳过 SA" % e)

mols = [m for m in (Chem.MolFromSmiles(s) for s in smis) if m is not None]
print("可解析分子: %d / %d" % (len(mols), len(smis)))
if mols:
    qeds = [QED.qed(m) for m in mols]
    mws = [Descriptors.MolWt(m) for m in mols]
    sas = [SA_CALC(m) for m in mols] if SA_CALC else []
    scafs = {MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols}
    print("QED  中位 %.3f  (范围 %.3f–%.3f)" % (np.median(qeds), min(qeds), max(qeds)))
    print("MW   中位 %.1f" % np.median(mws))
    if sas:
        print("SA   中位 %.2f" % np.median(sas))
        print("类药性综合分中位 %.3f" % np.median([q + (1 - s / 10) for q, s in zip(qeds, sas)]))
    print("唯一 Murcko 骨架: %d / %d" % (len(scafs), len(mols)))


In [ ]:
# 新颖性：与 A2A 已知活性集的最大 Tanimoto 相似度
KNOWN = {"A2A": "A2A_known_set.txt", "B2AR": "beta-2_adrenergic_receptor_known_set.txt",
         "D3": "dopamine_D3_receptor_known_set.txt", "5HT2B": "5-HT2B_receptor_known_set.txt"}
kf = []
p = os.path.join(KNOWN_DRUGS, KNOWN[TARGET])
for line in open(p, encoding="utf-8", errors="ignore"):
    m = Chem.MolFromSmiles(line.strip())
    if m is not None:
        kf.append(AllChem.GetMorganFingerprint(m, 2))
print("已知活性参考集: %d 个分子 (%s)" % (len(kf), os.path.relpath(p, PROJ)))
if mols and kf:
    sims = sorted(max(DataStructs.BulkTanimotoSimilarity(AllChem.GetMorganFingerprint(m, 2), kf))
                  for m in mols)
    print("最大 Tanimoto: 中位 %.3f  最小 %.3f  最大 %.3f" % (
        np.median(sims), sims[0], sims[-1]))
    print("提示: >=0.8 表示与已知活性高度相似，predict.py 会在备注中标注")


## 6. 一键入口速查

```bash
# 只做生成（可指定靶点、规模、引导强度）
python design.py --target A2A --num-samples 50 --beam 100 --seed 2024

# 一键生成最终候选清单 -> results/results.csv + results/structures/
python predict.py --target A2A --num-samples 50 --top 100

# 复用已生成的会话，只重跑筛选与排序
python predict.py --skip-design --runs outputs/design/A2A_<时间戳> --target A2A

# 训练/微调（自训练项目；本项目最终交付使用官方权重，见 ModelCard.md）
python train.py --config configs/train_multitarget_v2.yml
```

输出位置：

| 产物 | 路径 |
|---|---|
| 候选清单（含必填字段） | `results/results.csv`（UTF-8） |
| 候选三维结构 | `results/structures/*.sdf` |
| 运行清单（参数/种子/环境/耗时） | `results/run_manifest.json` |
| 入库全量分子及描述符 | `<outdir>/library/compounds.csv` |
| 训练日志与检查点 | `logs/<config>_<时间戳>/` |

**说明**：模型不输出亲和力/活性预测。清单中的类药性、新颖性指标由 RDKit 与
已知活性参考集计算，对接分数（若提供）来自 AutoDock Vina 这一独立第三方工具。
